In [60]:
# ==========================================================
# CELL 1 — IMPORTS
# ==========================================================

import pandas as pd
import sqlite3

from pathlib import Path

pd.set_option("display.max_columns", 100)

print("Imports successful.")

Imports successful.


In [61]:
# ==========================================================
# CELL 2 — PROJECT PATHS
# ==========================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebooks":
    BASE_DIR = CURRENT_DIR.parent
else:
    BASE_DIR = CURRENT_DIR

PROCESSED_DIR = BASE_DIR / "data" / "processed"
SQL_DIR = BASE_DIR / "sql"

SQL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Base directory:")
print(BASE_DIR)

print("\nProcessed directory:")
print(PROCESSED_DIR)

print("\nSQL directory:")
print(SQL_DIR)

Base directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics

Processed directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\data\processed

SQL directory:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql


In [62]:
# ==========================================================
# CELL 3 — CHECK PROCESSED FILES
# ==========================================================

required_files = [
    "players_features.csv",
    "teams_features.csv",
    "matches_features.csv"
]

print("Checking processed files...\n")

for filename in required_files:

    filepath = PROCESSED_DIR / filename

    if filepath.exists():
        print(f"✓ {filename}")
    else:
        print(f"✗ MISSING: {filename}")

Checking processed files...

✓ players_features.csv
✓ teams_features.csv
✓ matches_features.csv


In [63]:
# ==========================================================
# CELL 4 — LOAD PROCESSED DATA
# ==========================================================

players = pd.read_csv(
    PROCESSED_DIR / "players_features.csv"
)

teams = pd.read_csv(
    PROCESSED_DIR / "teams_features.csv"
)

matches = pd.read_csv(
    PROCESSED_DIR / "matches_features.csv"
)

print("Players shape:", players.shape)
print("Teams shape:", teams.shape)
print("Matches shape:", matches.shape)

Players shape: (1248, 80)
Teams shape: (48, 137)
Matches shape: (104, 53)


In [64]:
# ==========================================================
# CELL 5 — INSPECT COLUMNS
# ==========================================================

print("=" * 60)
print("PLAYER COLUMNS")
print("=" * 60)

print(players.columns.tolist())

print("\n" + "=" * 60)
print("TEAM COLUMNS")
print("=" * 60)

print(teams.columns.tolist())

print("\n" + "=" * 60)
print("MATCH COLUMNS")
print("=" * 60)

print(matches.columns.tolist())

PLAYER COLUMNS
['player', 'team', 'team_country', 'position', 'age', 'birth_year', 'club', 'games', 'games_starts', 'minutes', 'minutes_90s', 'goals', 'assists', 'goals_assists', 'goals_pens', 'pens_made', 'pens_att', 'cards_yellow', 'cards_red', 'goals_per90', 'assists_per90', 'goals_assists_per90', 'goals_pens_per90', 'goals_assists_pens_per90', 'shots', 'shots_on_target', 'shots_on_target_pct', 'shots_per90', 'shots_on_target_per90', 'goals_per_shot', 'goals_per_shot_on_target', 'minutes_per_game', 'minutes_pct', 'minutes_per_start', 'games_complete', 'games_subs', 'minutes_per_sub', 'unused_subs', 'points_per_game', 'on_goals_for', 'on_goals_against', 'plus_minus', 'plus_minus_per90', 'plus_minus_wowy', 'cards_yellow_red', 'fouls', 'fouled', 'offsides', 'crosses', 'interceptions', 'tackles_won', 'own_goals', 'gk_games', 'gk_games_starts', 'gk_minutes', 'gk_goals_against', 'gk_goals_against_per90', 'gk_shots_on_target_against', 'gk_saves', 'gk_save_pct', 'gk_wins', 'gk_ties', 'gk_lo

In [65]:
# ==========================================================
# CELL 6 — CREATE SQLITE DATABASE
# ==========================================================

DATABASE_PATH = SQL_DIR / "football_analytics.db"

connection = sqlite3.connect(
    DATABASE_PATH
)

print("SQLite database created/opened:")
print(DATABASE_PATH)

SQLite database created/opened:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql\football_analytics.db


In [66]:
# ==========================================================
# CELL 7 — CREATE SQL TABLES
# ==========================================================

players.to_sql(
    "players",
    connection,
    if_exists="replace",
    index=False
)

teams.to_sql(
    "teams",
    connection,
    if_exists="replace",
    index=False
)

matches.to_sql(
    "matches",
    connection,
    if_exists="replace",
    index=False
)

print("SQL tables created successfully.")

SQL tables created successfully.


In [67]:
# ==========================================================
# CELL 8 — VALIDATE SQL TABLES
# ==========================================================

tables = pd.read_sql_query(
    """
    SELECT name
    FROM sqlite_master
    WHERE type = 'table'
    ORDER BY name;
    """,
    connection
)

display(tables)

,name
0,matches
1,players
2,teams


In [68]:
# ==========================================================
# CELL 9 — TABLE ROW COUNTS
# ==========================================================

row_counts = pd.read_sql_query(
    """
    SELECT
        (SELECT COUNT(*) FROM players) AS players,
        (SELECT COUNT(*) FROM teams) AS teams,
        (SELECT COUNT(*) FROM matches) AS matches;
    """,
    connection
)

display(row_counts)

,players,teams,matches
0,1248,48,104


In [69]:
# ==========================================================
# CELL 10 — TOP SCORERS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    minutes
FROM players
WHERE goals > 0
ORDER BY goals DESC, assists DESC
LIMIT 20;
"""

top_scorers = pd.read_sql_query(
    query,
    connection
)

display(top_scorers)

,player,team,position,goals,assists,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,614.0
3,Erling Haaland,Norway,FW,7.0,0.0,465.0
4,Ousmane Dembélé,France,MF,6.0,2.0,592.0
5,Harry Kane,England,FW,6.0,1.0,652.0
6,Mikel Oyarzabal,Spain,FW,5.0,1.0,601.0
7,Vinicius Júnior,Brazil,FW,4.0,1.0,440.0
8,Julián Quiñones,Mexico,"FW,MF",4.0,1.0,410.0
9,Ismaila Sarr,Senegal,"MF,FW",4.0,1.0,364.0


In [70]:
# ==========================================================
# CELL 11 — TOP GOAL CONTRIBUTORS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    goals,
    assists,
    goal_contributions,
    minutes
FROM players
WHERE goal_contributions IS NOT NULL
ORDER BY goal_contributions DESC
LIMIT 20;
"""

top_contributors = pd.read_sql_query(
    query,
    connection
)

display(top_contributors)

,player,team,position,goals,assists,goal_contributions,minutes
0,Kylian Mbappé,France,FW,10.0,4.0,14.0,695.0
1,Lionel Messi,Argentina,FW,8.0,4.0,12.0,740.0
2,Jude Bellingham,England,MF,7.0,1.0,8.0,614.0
3,Ousmane Dembélé,France,MF,6.0,2.0,8.0,592.0
4,Harry Kane,England,FW,6.0,1.0,7.0,652.0
5,Michael Olise,France,MF,0.0,7.0,7.0,646.0
6,Erling Haaland,Norway,FW,7.0,0.0,7.0,465.0
7,Bukayo Saka,England,MF,3.0,3.0,6.0,358.0
8,Mikel Oyarzabal,Spain,FW,5.0,1.0,6.0,601.0
9,Vinicius Júnior,Brazil,FW,4.0,1.0,5.0,440.0


In [71]:
# ==========================================================
# CELL 12 — ATTACKING EFFICIENCY
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    minutes,
    goals_per90,
    assists_per90,
    goal_contributions_per90
FROM players
WHERE minutes >= 180
AND goal_contributions_per90 IS NOT NULL
ORDER BY goal_contributions_per90 DESC
LIMIT 20;
"""

efficient_attackers = pd.read_sql_query(
    query,
    connection
)

display(efficient_attackers)

,player,team,position,minutes,goals_per90,assists_per90,goal_contributions_per90
0,Johan Manzambi,Switzerland,"MF,FW",199.0,1.36,0.90,2.26
1,Kylian Mbappé,France,FW,695.0,1.29,0.52,1.81
2,Romelu Lukaku,Belgium,FW,233.0,1.16,0.39,1.55
3,Bukayo Saka,England,MF,358.0,0.75,0.75,1.50
4,Nathan Saliba,Canada,MF,182.0,0.49,0.99,1.48
5,Lionel Messi,Argentina,FW,740.0,0.97,0.49,1.46
6,Andreas Schjelderup,Norway,FW,252.0,0.36,1.07,1.43
7,Crysencio Summerville,Netherlands,FW,253.0,0.71,0.71,1.42
8,Erling Haaland,Norway,FW,465.0,1.35,0.00,1.35
9,Ismaila Sarr,Senegal,"MF,FW",364.0,0.99,0.25,1.24


In [72]:
# ==========================================================
# CELL 13 — PLAYER DEFENSIVE LEADERS
# ==========================================================

query = """
SELECT
    player,
    team,
    position,
    minutes,
    tackles_won,
    interceptions,
    defensive_actions,
    defensive_actions_per90
FROM players
WHERE minutes >= 180
AND defensive_actions_per90 IS NOT NULL
ORDER BY defensive_actions_per90 DESC
LIMIT 20;
"""

defensive_leaders = pd.read_sql_query(
    query,
    connection
)

display(defensive_leaders)

,player,team,position,minutes,tackles_won,interceptions,defensive_actions,defensive_actions_per90
0,Marvin Senaya,Ghana,DF,278.0,12.0,6.0,18.0,5.806
1,Rayan Aït-Nouri,Algeria,DF,284.0,13.0,5.0,18.0,5.625
2,Aurélien Tchouaméni,France,MF,360.0,14.0,6.0,20.0,5.000
3,Merchas Doski,Iraq,DF,270.0,9.0,6.0,15.0,5.000
4,Mohanad Lasheen,Egypt,MF,359.0,11.0,8.0,19.0,4.750
5,Khuliso Mudau,South Africa,DF,360.0,8.0,11.0,19.0,4.750
6,Livano Comenencia,Curaçao,MF,233.0,1.0,11.0,12.0,4.615
7,Mohamed Amine Ben Hamida,Tunisia,DF,202.0,9.0,1.0,10.0,4.545
8,Andrés Cubas,Paraguay,MF,480.0,13.0,11.0,24.0,4.528
9,Diego Gómez,Paraguay,MF,304.0,6.0,9.0,15.0,4.412


In [73]:
# ==========================================================
# CELL 14 — TEAM PERFORMANCE
# ==========================================================

query = """
SELECT
    team,
    games,
    goals,
    assists,
    shots,
    shots_on_target,
    goals_per90
FROM teams
ORDER BY goals DESC
LIMIT 20;
"""

team_performance = pd.read_sql_query(
    query,
    connection
)

display(team_performance)

,team,games,goals,assists,shots,shots_on_target,goals_per90
0,England,8,20,14,118,53,2.40
1,France,8,20,18,139,59,2.50
2,Argentina,8,18,12,114,44,2.00
3,Belgium,6,13,10,112,34,2.05
4,Spain,8,13,10,140,54,1.56
5,Norway,6,12,10,66,29,1.89
6,Germany,4,11,10,74,28,2.54
7,Brazil,5,10,8,74,30,2.00
8,Mexico,5,10,7,70,21,2.00
9,Morocco,6,10,9,69,26,1.58


In [74]:
# ==========================================================
# CELL 15 — TEAM ATTACKING EFFICIENCY
# ==========================================================

query = """
SELECT
    team,
    goals,
    shots,
    shots_on_target,
    goals_per_shot
FROM teams
WHERE goals_per_shot IS NOT NULL
ORDER BY goals_per_shot DESC
LIMIT 20;
"""

team_efficiency = pd.read_sql_query(
    query,
    connection
)

display(team_efficiency)

,team,goals,shots,shots_on_target,goals_per_shot
0,Japan,8,34,13,0.2353
1,Netherlands,10,46,22,0.2174
2,Norway,12,66,29,0.1818
3,England,20,118,53,0.1695
4,Croatia,6,37,17,0.1622
5,Argentina,18,114,44,0.1579
6,Austria,5,32,8,0.1562
7,United States,9,59,19,0.1525
8,Germany,11,74,28,0.1486
9,Sweden,7,48,23,0.1458


In [75]:
# ==========================================================
# CELL 16 — TEAM DEFENSIVE ACTIVITY
# ==========================================================

query = """
SELECT
    team,
    tackles_won,
    interceptions,
    defensive_actions
FROM teams
WHERE defensive_actions IS NOT NULL
ORDER BY defensive_actions DESC
LIMIT 20;
"""

team_defense = pd.read_sql_query(
    query,
    connection
)

display(team_defense)

,team,tackles_won,interceptions,defensive_actions
0,Argentina,93,82,175
1,Spain,88,64,152
2,France,77,65,142
3,Paraguay,88,53,141
4,England,77,51,128
5,Norway,67,44,111
6,Switzerland,52,52,104
7,United States,42,60,102
8,Morocco,61,40,101
9,Egypt,54,44,98


In [76]:
# ==========================================================
# CELL 17 — MATCH RESULT DISTRIBUTION
# ==========================================================

query = """
SELECT
    result,
    COUNT(*) AS matches,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM matches),
        2
    ) AS percentage
FROM matches
GROUP BY result
ORDER BY matches DESC;
"""

match_results = pd.read_sql_query(
    query,
    connection
)

display(match_results)

,result,matches,percentage
0,Home Win,50,48.08
1,Away Win,30,28.85
2,Draw,24,23.08


In [77]:
# ==========================================================
# CELL 18 — MATCH PERFORMANCE DRIVERS
# ==========================================================

query = """
SELECT
    result,
    COUNT(*) AS matches,
    ROUND(AVG(total_goals), 2) AS avg_total_goals,
    ROUND(AVG(possession_difference), 2) AS avg_possession_difference,
    ROUND(AVG(shot_difference), 2) AS avg_shot_difference,
    ROUND(AVG(shots_on_target_difference), 2) AS avg_sot_difference
FROM matches
GROUP BY result;
"""

match_drivers = pd.read_sql_query(
    query,
    connection
)

display(match_drivers)

,result,matches,avg_total_goals,avg_possession_difference,avg_shot_difference,avg_sot_difference
0,Away Win,30,3.13,-13.20,-2.60,-2.60
1,Draw,24,1.50,11.92,2.42,0.33
2,Home Win,50,3.44,12.02,7.16,3.84


In [78]:
# ==========================================================
# CELL 19 — PLAYER + TEAM JOIN
# ==========================================================

query = """
SELECT
    p.player,
    p.team,
    p.position,
    p.goals,
    p.assists,
    p.goal_contributions,
    t.goals AS team_goals,
    t.possession
FROM players AS p
INNER JOIN teams AS t
    ON p.team = t.team
ORDER BY p.goal_contributions DESC
LIMIT 20;
"""

player_team_join = pd.read_sql_query(
    query,
    connection
)

display(player_team_join)

,player,team,position,goals,assists,goal_contributions,team_goals,possession
0,Kylian Mbappé,France,FW,10.0,4.0,14.0,20,55.9
1,Lionel Messi,Argentina,FW,8.0,4.0,12.0,18,57.6
2,Jude Bellingham,England,MF,7.0,1.0,8.0,20,54.1
3,Ousmane Dembélé,France,MF,6.0,2.0,8.0,20,55.9
4,Harry Kane,England,FW,6.0,1.0,7.0,20,54.1
5,Michael Olise,France,MF,0.0,7.0,7.0,20,55.9
6,Erling Haaland,Norway,FW,7.0,0.0,7.0,12,52.2
7,Bukayo Saka,England,MF,3.0,3.0,6.0,20,54.1
8,Mikel Oyarzabal,Spain,FW,5.0,1.0,6.0,13,64.1
9,Vinicius Júnior,Brazil,FW,4.0,1.0,5.0,10,53.0


In [79]:
# ==========================================================
# CELL 20 — SAVE SQL ANALYSIS RESULTS
# ==========================================================

sql_results = {
    "top_scorers": top_scorers,
    "top_contributors": top_contributors,
    "efficient_attackers": efficient_attackers,
    "defensive_leaders": defensive_leaders,
    "team_performance": team_performance,
    "team_efficiency": team_efficiency,
    "team_defense": team_defense,
    "match_results": match_results,
    "match_drivers": match_drivers,
    "player_team_join": player_team_join
}

print("Saving SQL analysis results...\n")

for name, dataframe in sql_results.items():

    output_path = SQL_DIR / f"{name}.csv"

    dataframe.to_csv(
        output_path,
        index=False
    )

    print(f"✓ {output_path.name}")

print("\nAll SQL analysis results saved.")

Saving SQL analysis results...

✓ top_scorers.csv
✓ top_contributors.csv
✓ efficient_attackers.csv
✓ defensive_leaders.csv
✓ team_performance.csv
✓ team_efficiency.csv
✓ team_defense.csv
✓ match_results.csv
✓ match_drivers.csv
✓ player_team_join.csv

All SQL analysis results saved.


In [80]:
# ==========================================================
# CELL 21 — FINAL VALIDATION
# ==========================================================

print("=" * 70)
print("PART 05 — SQL VALIDATION COMPLETE")
print("=" * 70)

print("\nDatabase:")
print(DATABASE_PATH)

print("\nSQL Tables:")
display(tables)

print("\nSQL Analysis Outputs:")

for name in sql_results:

    filepath = SQL_DIR / f"{name}.csv"

    if filepath.exists():
        print(f"✓ {filepath.name}")
    else:
        print(f"✗ Missing: {filepath.name}")

print("\nNumber of SQL analyses:", len(sql_results))

# Close database LAST
connection.close()

print("\nSQLite connection closed.")
print("\n✅ PART 05 COMPLETE")

PART 05 — SQL VALIDATION COMPLETE

Database:
c:\Users\AYUSH SINGH\OneDrive\Desktop\Dispersion\resume projects\FIFA-World-Cup-Analytics\sql\football_analytics.db

SQL Tables:


,name
0,matches
1,players
2,teams



SQL Analysis Outputs:
✓ top_scorers.csv
✓ top_contributors.csv
✓ efficient_attackers.csv
✓ defensive_leaders.csv
✓ team_performance.csv
✓ team_efficiency.csv
✓ team_defense.csv
✓ match_results.csv
✓ match_drivers.csv
✓ player_team_join.csv

Number of SQL analyses: 10

SQLite connection closed.

✅ PART 05 COMPLETE
